# AEGIS — Fine-tuning the prompt-injection detector

Continues fine-tuning `protectai/deberta-v3-base-prompt-injection-v2` on direct **and indirect** (BIPIA-style) injection, so the detector stops missing attacks hidden inside emails and tables.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).

Steps: 1) setup · 2) load data · 3) document-level split · 4) crop around attack · 5) balance · 6) tokenize · 7) train · 8) validate + pick threshold · 9) save · 10) held-out test vs baseline

In [ ]:
# Colab already ships pyarrow, torch, pandas and scikit-learn; installing only the extras avoids version conflicts
!pip -q install transformers datasets sentencepiece

In [ ]:
import os, json, time, random, hashlib, math
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## Config

Put your parquet files in Google Drive (e.g. `MyDrive/aegis/data/`) or upload them to `/content/data/`.
If `SAFEGUARD_PATH` / `DEEPSET_PATH` don't exist, the notebook pulls them from the Hugging Face Hub instead (Colab's connection is fast).
BIPIA has to come from your own parquet.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/aegis/data"
    OUT_DIR  = "/content/drive/MyDrive/aegis/models/aegis-detector"
else:
    DATA_DIR = "/content/data"
    OUT_DIR  = "/content/models/aegis-detector"

SAFEGUARD_PATH = f"{DATA_DIR}/safe-guard-train.parquet"   # label: 1 = injection
DEEPSET_PATH   = f"{DATA_DIR}/deepset-train.parquet"      # label: 1 = injection
BIPIA_PATH     = f"{DATA_DIR}/bipia.parquet"              # columns: context, attack_str (+ task/type)
SUBTLE_PATH    = f"{DATA_DIR}/subtle.json"                # optional: [{"text": ..., "label": 0/1}]

MODEL_ID   = "protectai/deberta-v3-base-prompt-injection-v2"
MAX_LEN    = 256
BATCH      = 16
EPOCHS     = 2
LR         = 2e-5
CROP_CHARS = 1500
N_SAFEGUARD = 1500
N_BIPIA     = 2000   # half attacked, half clean
MAX_FA      = 0.02   # max false-alarm rate when choosing the threshold
os.makedirs(OUT_DIR, exist_ok=True)

## 0. Build `bipia.parquet` from Microsoft's BIPIA repo

BIPIA ships as clean email/table documents plus attack strings, not as a parquet. This cell inserts attacks (start / middle / end) into each document.
Held-out documents (md5 bucket 0) only receive attacks from `text_attack_test.json`, so the test set has unseen documents **and** unseen attacks.
Skipped if the parquet already exists.

In [ ]:
import subprocess

def read_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

def flatten_attacks(p):
    data = json.load(open(p))                      # {category: [attack strings]} or a list
    if isinstance(data, dict):
        return [a for v in data.values() for a in (v if isinstance(v, list) else [v])]
    return list(data)

def get_context(row):
    for k in ["context", "email", "table", "document"]:
        if k in row and isinstance(row[k], str):
            return row[k]
    raise KeyError(f"no context field in {list(row)}")

def insert_attack(ctx, atk, pos):
    if pos == "start":
        return atk + "\n" + ctx
    if pos == "end":
        return ctx + "\n" + atk
    lines = ctx.split("\n")                        # middle: on a line boundary
    k = len(lines) // 2
    return "\n".join(lines[:k] + [atk] + lines[k:])

if os.path.exists(BIPIA_PATH):
    print("found", BIPIA_PATH, "- skipping build")
else:
    if not os.path.exists("/content/BIPIA"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/microsoft/BIPIA.git", "/content/BIPIA"], check=True)
    B = "/content/BIPIA/benchmark"
    atk_train = flatten_attacks(f"{B}/text_attack_train.json")
    atk_test  = flatten_attacks(f"{B}/text_attack_test.json")
    print("attacks:", len(atk_train), "train /", len(atk_test), "test")

    brng = random.Random(SEED)
    ATTACKS_PER_DOC = 6                            # caps size; later cells sample N_BIPIA anyway
    rows = []
    for task in ["email", "table"]:
        for split_file in ["train", "test"]:
            p = f"{B}/{task}/{split_file}.jsonl"
            if not os.path.exists(p):
                print("missing", p); continue
            for r in read_jsonl(p):
                ctx = get_context(r)
                bucket = int(hashlib.md5(ctx.encode("utf-8")).hexdigest(), 16) % 10
                pool = atk_test if bucket == 0 else atk_train
                for atk in brng.sample(pool, min(ATTACKS_PER_DOC, len(pool))):
                    pos = brng.choice(["start", "middle", "end"])
                    rows.append({"task": task, "context": insert_attack(ctx, atk, pos),
                                 "attack_str": atk, "position": pos})
    bipia_df = pd.DataFrame(rows)
    os.makedirs(DATA_DIR, exist_ok=True)
    bipia_df.to_parquet(BIPIA_PATH, index=False)
    print(bipia_df.groupby("task").size().to_dict(), "->", BIPIA_PATH)

## 1. Load data → `{text, label}`

In [ ]:
def load_labeled(path, hf_name):
    if os.path.exists(path):
        df = pd.read_parquet(path)
    else:
        from datasets import load_dataset
        print(f"{path} not found → downloading {hf_name}")
        df = load_dataset(hf_name, split="train").to_pandas()
    df = df.rename(columns={"prompt": "text"})[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    return df

safeguard = load_labeled(SAFEGUARD_PATH, "xTRam1/safe-guard-prompt-injection")
deepset   = load_labeled(DEEPSET_PATH,   "deepset/prompt-injections")
print("safe-guard:", len(safeguard), safeguard.label.value_counts().to_dict())
print("deepset:   ", len(deepset),   deepset.label.value_counts().to_dict())

bipia_raw = pd.read_parquet(BIPIA_PATH)
print("BIPIA columns:", list(bipia_raw.columns), "rows:", len(bipia_raw))

# keep email + table rows if a task/type column exists
task_col = next((c for c in ["task", "type", "source", "domain"] if c in bipia_raw.columns), None)
if task_col:
    bipia_raw = bipia_raw[bipia_raw[task_col].astype(str).str.lower().str.contains("email|table")]
bipia_raw = bipia_raw[[a in c for a, c in zip(bipia_raw.attack_str.astype(str), bipia_raw.context.astype(str))]]
print("BIPIA rows with attack_str in context:", len(bipia_raw))

subtle = None
if os.path.exists(SUBTLE_PATH):
    subtle = pd.DataFrame(json.load(open(SUBTLE_PATH)))[["text", "label"]]
    print("subtle:", len(subtle))

## 2–3. Split BIPIA by document, crop around the attack

Doc id = **stable** hash (md5) of the clean text. Python's built-in `hash()` is randomized per process, so don't use it — `benchmark.py` must use the same function to get the same split.

In [ ]:
def doc_bucket(clean_text: str) -> int:
    return int(hashlib.md5(clean_text.encode("utf-8")).hexdigest(), 16) % 10

def split_of(clean_text):
    b = doc_bucket(clean_text)
    return "test" if b == 0 else "val" if b == 1 else "train"

def crop_pair(context, attack, n=CROP_CHARS):
    """Crop ~n chars around the attack; crop the clean twin at the same spot."""
    i = context.find(attack)
    clean_full = context[:i] + context[i + len(attack):]   # remove this occurrence only
    if len(context) <= n:
        return context, clean_full
    n = max(n, len(attack) + 200)                            # window must hold the whole attack
    start = max(0, i + len(attack) // 2 - n // 2)
    start = min(start, i, max(0, len(context) - n))
    attacked = context[start:start + n]
    clean = clean_full[start:start + n]                      # same start; text before `start` is identical
    return attacked, clean

rows = []
for ctx, atk in zip(bipia_raw.context.astype(str), bipia_raw.attack_str.astype(str)):
    clean_full = ctx.replace(atk, "")
    sp = split_of(clean_full)
    a, c = crop_pair(ctx, atk)
    rows.append({"text": a, "label": 1, "split": sp, "doc": clean_full})
    rows.append({"text": c, "label": 0, "split": sp, "doc": clean_full})
bipia = pd.DataFrame(rows)
print(bipia.groupby(["split", "label"]).size().unstack())

In [ ]:
# Paste this into simulator/benchmark.py and use it behind a --heldout flag:
print('''
import hashlib
def is_heldout(clean_text: str) -> bool:
    return int(hashlib.md5(clean_text.encode("utf-8")).hexdigest(), 16) % 10 == 0
''')

## 4. Balance, cap, shuffle

In [ ]:
rng = random.Random(SEED)

def sample(df, n):
    return df.sample(n=min(n, len(df)), random_state=SEED)

def split_generic(df, name):
    # direct-injection sets: hold out 10% for validation (by row; these are single prompts)
    idx = list(range(len(df))); rng.shuffle(idx)
    cut = max(1, len(idx) // 10)
    df = df.reset_index(drop=True).copy()
    df["split"] = "train"; df.loc[idx[:cut], "split"] = "val"; df["src"] = name
    return df

sg = split_generic(sample(safeguard, N_SAFEGUARD), "safeguard")
ds = split_generic(deepset, "deepset")
parts = [sg, ds]
if subtle is not None:
    parts.append(split_generic(subtle, "subtle"))

b_train = bipia[bipia.split == "train"]
per_class = N_BIPIA // 2
b_train = pd.concat([sample(b_train[b_train.label == 1], per_class),
                     sample(b_train[b_train.label == 0], per_class)])
b_val_all = bipia[bipia.split == "val"]
n_val = N_BIPIA // 8 // 2                      # per class -> 250 total, mirrors 80/10 vs the 2,000 train
b_val = pd.concat([sample(b_val_all[b_val_all.label == 1], n_val),
                   sample(b_val_all[b_val_all.label == 0], n_val)]).copy()
b_test = bipia[bipia.split == "test"].copy()
for d in (b_train, b_val, b_test):
    d["src"] = "bipia"
parts += [b_train, b_val]

all_df = pd.concat(parts, ignore_index=True)[["text", "label", "split", "src"]]
train_df = all_df[all_df.split == "train"].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = all_df[all_df.split == "val"].reset_index(drop=True)
test_bipia = b_test[["text", "label", "src"]].reset_index(drop=True)

print("train:", len(train_df), train_df.groupby("src").label.value_counts().unstack().to_dict())
print("val:  ", len(val_df), "| BIPIA held-out test:", len(test_bipia))

## 5–6. Tokenizer + model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
print("label2id:", model.config.label2id)
INJ = model.config.label2id.get("INJECTION", 1)
assert model.config.num_labels == 2
if INJ != 1:
    print("NOTE: INJECTION id is", INJ, "→ remapping our labels")

In [ ]:
class TextDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.t = df.text.astype(str).tolist()
        self.y = [INJ if l == 1 else 1 - INJ for l in df.label.tolist()]
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]

def collate(batch):
    texts, ys = zip(*batch)
    enc = tokenizer(list(texts), max_length=MAX_LEN, truncation=True, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(ys)
    return enc

train_dl = torch.utils.data.DataLoader(TextDS(train_df), batch_size=BATCH, shuffle=True,
                                       collate_fn=collate, generator=torch.Generator().manual_seed(SEED))

@torch.no_grad()
def predict(df, mdl, bs=64):
    mdl.eval(); out = []
    texts = df.text.astype(str).tolist()
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_LEN, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = mdl(**enc).logits
        out.append(torch.softmax(logits.float(), -1)[:, INJ].cpu().numpy())
    return np.concatenate(out)

def metrics(y, p, thr=0.5):
    y = np.asarray(y); pred = (p >= thr).astype(int)
    det = pred[y == 1].mean() if (y == 1).any() else float("nan")
    fa  = pred[y == 0].mean() if (y == 0).any() else float("nan")
    auc = roc_auc_score(y, p) if len(set(y)) == 2 else float("nan")
    return {"detection_rate": float(det), "false_alarm_rate": float(fa),
            "f1": float(f1_score(y, pred, zero_division=0)), "auc": float(auc)}

### Baseline (before training) — for comparison

In [ ]:
base_val  = predict(val_df, model)
base_test = predict(test_bipia, model)
baseline = {"val": metrics(val_df.label, base_val),
            "bipia_heldout": metrics(test_bipia.label, base_test)}
print(json.dumps(baseline, indent=2))

## 7–8. Train (plain PyTorch) + validate each epoch

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

best = {"f1": -1}
history = []
t0 = time.time()
for epoch in range(EPOCHS):
    model.train(); running = 0.0; te = time.time()
    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad()
        running += loss.item()
        if step % 50 == 0:
            el = time.time() - te
            print(f"epoch {epoch+1} step {step}/{len(train_dl)}  loss {running/50:.4f}  "
                  f"({el:.0f}s, ~{el/step*len(train_dl)/60:.1f} min/epoch)")
            running = 0.0

    p_val = predict(val_df, model)
    m = metrics(val_df.label, p_val)
    m_bipia = metrics(val_df.label[val_df.src == "bipia"], p_val[val_df.src.values == "bipia"])
    history.append({"epoch": epoch + 1, "val": m, "val_bipia": m_bipia, "secs": time.time() - te})
    print(f"\n== epoch {epoch+1}: val {m}\n   val BIPIA {m_bipia}\n")
    if m["f1"] > best["f1"]:
        best = {"f1": m["f1"], "epoch": epoch + 1, "p_val": p_val}
        model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
        print("   saved as best so far")

train_secs = time.time() - t0
print(f"training time: {train_secs/60:.1f} min, best epoch {best['epoch']}")

### Choose the threshold on validation: highest detection with false alarms < 2%

In [ ]:
y_val = val_df.label.values; p_val = best["p_val"]
cands = []
for thr in np.linspace(0.05, 0.99, 95):
    m = metrics(y_val, p_val, thr)
    if m["false_alarm_rate"] < MAX_FA:
        cands.append((m["detection_rate"], thr, m))
if cands:
    _, THRESHOLD, thr_metrics = max(cands, key=lambda c: (c[0], c[1]))
else:
    THRESHOLD, thr_metrics = 0.5, metrics(y_val, p_val, 0.5)
    print(f"no threshold keeps false alarms < {MAX_FA:.0%}; falling back to 0.5")
print("threshold:", round(float(THRESHOLD), 3), thr_metrics)

## 9–10. Test both models on held-out sets + save `metrics.json`

Three test sets: held-out BIPIA (indirect, unseen docs + unseen attacks) and the official safe-guard / deepset test splits (direct — checks nothing regressed).
This table is the section 6.3 results table.

In [ ]:
from datasets import load_dataset

def hf_test(name):
    df = load_dataset(name, split="test").to_pandas().rename(columns={"prompt": "text"})[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    return df

test_sets = {
    "bipia_heldout (indirect)": test_bipia,
    "safe-guard test (direct)": hf_test("xTRam1/safe-guard-prompt-injection"),
    "deepset test (direct)":    hf_test("deepset/prompt-injections"),
}

base_model  = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
tuned_model = AutoModelForSequenceClassification.from_pretrained(OUT_DIR).to(device)   # best epoch

results, table = {}, []
for name, df in test_sets.items():
    pb, pt = predict(df, base_model), predict(df, tuned_model)
    for model_name, p, thr in [("baseline (ProtectAI)", pb, 0.5),
                               ("fine-tuned", pt, 0.5),
                               ("fine-tuned", pt, THRESHOLD)]:
        m = metrics(df.label, p, thr)
        key = f"{model_name} @{thr:.2f}"
        results.setdefault(name, {"n": int(len(df)), "n_injection": int(df.label.sum())})[key] = m
        table.append({"test set": name, "model": key, "n": len(df), **{k: round(v, 3) for k, v in m.items()}})
del base_model; torch.cuda.empty_cache()

results_df = pd.DataFrame(table).set_index(["test set", "model"])
display(results_df)

In [ ]:
metrics_out = {
    "base_model": MODEL_ID,
    "approach": "continued fine-tuning of the ProtectAI baseline on direct + indirect (BIPIA) injection",
    "dataset_counts": {
        "train_total": int(len(train_df)),
        "train_by_source": {k: int(v) for k, v in train_df.src.value_counts().items()},
        "val_total": int(len(val_df)),
        "val_by_source": {k: int(v) for k, v in val_df.src.value_counts().items()},
        "test_by_set": {k: int(len(v)) for k, v in test_sets.items()},
    },
    "split_rule": ("BIPIA split by document: md5(clean_text) % 10 -> 0 test, 1 val, else train; "
                   "test documents receive only text_attack_test.json attacks; "
                   f"BIPIA train capped at {N_BIPIA}, val at {2*n_val} (class-balanced), test uncapped. "
                   "safe-guard/deepset: 10% of train rows held out for val; official HF test splits for testing."),
    "hyperparams": {"epochs": EPOCHS, "max_len": MAX_LEN, "batch": BATCH, "lr": LR,
                    "weight_decay": 0.01, "warmup": 0.1, "crop_chars": CROP_CHARS, "seed": SEED},
    "epochs_trained": EPOCHS,
    "best_epoch": best["epoch"],
    "hardware": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "training_time_min": round(train_secs / 60, 2),
    "threshold": round(float(THRESHOLD), 3),
    "threshold_rule": f"highest validation detection rate with false-alarm rate < {MAX_FA:.0%}",
    "val_at_threshold": thr_metrics,
    "history": history,
    "results": results,          # section 6.3 table: detection / false alarms / F1 / AUC per test set
}
json.dump(metrics_out, open(f"{OUT_DIR}/metrics.json", "w"), indent=2)
print(json.dumps(metrics_out, indent=2))

## Get the model onto your laptop

If you used Drive, it's already in `MyDrive/aegis/models/aegis-detector` — sync it down with the Drive app.
Otherwise zip and download (~700 MB; the Drive route is gentler on a slow connection since it can resume).

Then in your repo:
```bash
echo "models/" >> .gitignore
python simulator/benchmark.py --heldout                                        # baseline
CLASSIFIER_MODEL=models/aegis-detector python simulator/benchmark.py --heldout  # yours
python simulator/runner.py --replay                                            # 6 scenarios still pass
```

In [ ]:
if not USE_DRIVE:
    !cd /content/models && zip -qr aegis-detector.zip aegis-detector
    from google.colab import files
    files.download("/content/models/aegis-detector.zip")